In [14]:
# Install Librosa
!pip install -q "librosa"

In [15]:
# Base folder
folder = '/content/gdrive/MyDrive/EAD26/'

# import
from google.colab import drive
# mount
drive.mount('/content/gdrive/', force_remount=True)

# /content/gdrive/MyDrive/EAD26/media/potter.wav

!ln -sf f"{folder}/media/" "/content/media/"

Mounted at /content/gdrive/
ln: failed to create symbolic link '/content/media/': No such file or directory


In [16]:
import librosa
import librosa.display
import soundfile as sf

import numpy as np
import scipy

from matplotlib import pyplot as plt
%matplotlib inline

import IPython
from pprint import pprint


In [17]:
def band_pass_filter(audio, sr, low_freq, high_freq):
    nyquist = sr / 2
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = scipy.signal.butter(2, [low, high], btype='band')
    return scipy.signal.filtfilt(b, a, audio)

# Keep only frequencies between 300Hz and 3000Hz
# band_passed = band_pass_filter(y, sr, 300, 3000)

In [19]:
# PySoundFile

# available subtypes for output audio coding
print("Codecs")
pprint(sf.available_subtypes())

# available formats for output audio container
print("Containers")
pprint(sf.available_formats())

Codecs
{'ALAC_16': '16 bit ALAC',
 'ALAC_20': '20 bit ALAC',
 'ALAC_24': '24 bit ALAC',
 'ALAC_32': '32 bit ALAC',
 'ALAW': 'A-Law',
 'DOUBLE': '64 bit float',
 'DPCM_16': '16 bit DPCM',
 'DPCM_8': '8 bit DPCM',
 'DWVW_12': '12 bit DWVW',
 'DWVW_16': '16 bit DWVW',
 'DWVW_24': '24 bit DWVW',
 'FLOAT': '32 bit float',
 'G721_32': '32kbs G721 ADPCM',
 'G723_24': '24kbs G723 ADPCM',
 'G723_40': '40kbs G723 ADPCM',
 'GSM610': 'GSM 6.10',
 'IMA_ADPCM': 'IMA ADPCM',
 'MPEG_LAYER_I': 'MPEG Layer I',
 'MPEG_LAYER_II': 'MPEG Layer II',
 'MPEG_LAYER_III': 'MPEG Layer III',
 'MS_ADPCM': 'Microsoft ADPCM',
 'NMS_ADPCM_16': '16kbs NMS ADPCM',
 'NMS_ADPCM_24': '24kbs NMS ADPCM',
 'NMS_ADPCM_32': '32kbs NMS ADPCM',
 'OPUS': 'Opus',
 'PCM_16': 'Signed 16 bit PCM',
 'PCM_24': 'Signed 24 bit PCM',
 'PCM_32': 'Signed 32 bit PCM',
 'PCM_S8': 'Signed 8 bit PCM',
 'PCM_U8': 'Unsigned 8 bit PCM',
 'ULAW': 'U-Law',
 'VORBIS': 'Vorbis',
 'VOX_ADPCM': 'VOX ADPCM'}
Containers
{'AIFF': 'AIFF (Apple/SGI)',
 'AU': 

In [21]:
def quantize_uniform(x, N=8):
  qmin, qmax, qlevel = -1, +1, 2**N-1 # defaults
  qstep = (qmax-qmin) / qlevel
  xnorm = (x-qmin) * qlevel / (qmax-qmin)
  xnorm[xnorm > qlevel] = qlevel
  xnorm[xnorm < 0] = 0
  xnorm_quant = np.floor(xnorm)
  xquant = xnorm_quant * (qmax-qmin) / qlevel
  xquant = xquant + qmin + qstep/2
  return xquant

def plot_vs_time(w, t, xlim=None, ylim=[-1, +1], fig=None):
  fig, ax = plt.subplots(figsize=(12,3)) if fig is None else fig
  ax.plot(t,w,'-')
  ax.set_xlabel('Time (s)')
  ax.set_ylabel('Amplitude')
  ax.set_xlim(xlim) if xlim is not None else plt.xlim(t[0],t[-1])
  ax.set_ylim(ylim) if ylim is not None else None
  fig.tight_layout()
  # ax.grid()
  # plt.show()

def SNR(original, quantized):
  noise = quantized - original
  powS = np.sum(original**2)
  powN = np.sum(noise**2)
  return 10*np.log10(powS/powN)

def normalize(x):
  return x / np.max(np.abs(x))

def RMS(x):
  return np.sqrt(np.mean(x**2))

In [20]:
# Narrowband 300-3400 Hz filtering

in_filename = f"{folder}/media/potter.wav"
out_filename = f"{folder}/media/potter.nb.wav"

# in_filename = f"{folder}/media/viper.mp3"

y, sr = librosa.load(in_filename, sr=None)
y_nb = band_pass_filter(y, sr, 300, 3400)

IPython.display.display( IPython.display.Audio(y,rate=sr) )
IPython.display.display(IPython.display.Audio(y_nb,rate=sr) )

sf.write(out_filename, y, sr, subtype='PCM_16')


In [28]:
# Low bitrate quantization

N = 2
in_filename = f"{folder}/media/potter.wav"
out_filename = f"{folder}/media/potter.{N:02d}bit.wav"

# in_filename = f"{folder}/media/viper.mp3"

y, sr = librosa.load(in_filename, sr=None)

# for N = 2 since the signal level may be too low and result in silence
# y = normalize(y)

y_nb = quantize_uniform(y, N)

IPython.display.display( IPython.display.Audio(y,rate=sr) )
IPython.display.display(IPython.display.Audio(y_nb,rate=sr) )

sf.write(out_filename, y, sr, subtype='PCM_16')
